[![View on GitHub](https://img.shields.io/badge/View_on-GitHub-181717?logo=github)](https://github.com/Skquark/AEI-Colab-Notebooks/blob/main/WildGaussianSplatting_Colab.ipynb)  [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/WildGaussianSplatting_Colab.ipynb)


# 🎥 Wild Gaussian Splatting — Video / Image Folder → 3DGS (CC BY-NC-SA + INRIA)

> **⚠️ License notice (please read before running):** This notebook is for **research and
> evaluation only**. The 3D Gaussian Splatting training code is the original
> [INRIA 3DGS](https://github.com/graphdeco-inria/gaussian-splatting) implementation,
> which is released under the **INRIA non-commercial research license** — commercial use
> requires written consent from Inria. The MASt3R backbone (used for pose + point cloud
> estimation) is licensed **CC BY-NC-SA 4.0** by Naver Corporation. The wrapper notebook
> itself is the same Apache-2.0 as the upstream [nerlfield/wild-gaussian-splatting](https://github.com/nerlfield/wild-gaussian-splatting)
> (no license file in that repo, but the README refers to the wrapper as Apache-2.0).
> See "License" below for the full breakdown.

**Wild Gaussian Splatting** (Daniel Kovalenko, reface.ai) is the closest open-source
equivalent to **Luma Labs Capture / Genie**: upload a casual video or a folder of
overlapping photos, get a polished 3DGS scene back. It chains [MASt3R](https://github.com/naver/mast3r)
(Naver, ICLR 2025) for pose + point cloud estimation with the original
[INRIA 3DGS](https://github.com/graphdeco-inria/gaussian-splatting) training loop,
producing a real 3DGS scene (not just novel-view video) in 5-15 minutes on a T4.

* **Project:** [github.com/nerlfield/wild-gaussian-splatting](https://github.com/nerlfield/wild-gaussian-splatting)
* **MASt3R (pose + point cloud):** [github.com/naver/mast3r](https://github.com/naver/mast3r) (CC BY-NC-SA 4.0)
* **3DGS (training):** [github.com/graphdeco-inria/gaussian-splatting](https://github.com/graphdeco-inria/gaussian-splatting) (INRIA non-commercial)
* **Reference notebooks:** `00_mast3r_inference.ipynb` + `01_gaussian_splatting_fitting.ipynb` in the upstream repo

## How it works

1. **Frame extraction** (if you uploaded a video) → save to a folder
2. **MASt3R alignment** — runs `sparse_global_alignment` on the input images, jointly
   optimizing camera poses, intrinsics, depth, and per-image point clouds via the
   kinematic-chain hierarchical-clustering scene graph
3. **COLMAP-format export** — write `cameras.txt`, `images.txt`, `points3D.ply` so the
   3DGS loader can read them
4. **3DGS training** — 3,000-30,000 iterations of the original INRIA densify/prune loop
5. **Render an orbit video** — MP4 of the final scene (using the upstream's
   `generate_ellipse_path_from_camera_infos` to interpolate cameras along an ellipse
   through the training-view centroid)
6. **Output:** `point_cloud.ply` (real 3DGS) + `cameras.json` + `renders.mp4` + per-frame PNGs

## When to use this vs NoPoSplat

* **NoPoSplat** — 2-3 unposed photos → real 3DGS in ~10 seconds. MIT-licensed notebook; output is yours. No INRIA / non-commercial baggage.
* **Wild Gaussian Splatting (this notebook)** — video / image folder → real 3DGS in 5-15 min with proper per-scene optimization. Higher fidelity than NoPoSplat but inherits non-commercial licenses. Best for hero / portfolio assets.
* **TripoSPlat** — single image → 3DGS via learned generative prior. Best for imagined / single-photo scenes. MIT, fully commercial-OK.

## Pipeline
```
Wild Gaussian Splatting (this notebook)  →  .ply + .mp4
                                                ↓
                                  SplatTransform_Colab STEP 3  →  SOG/SPLAT/SPZ/GLB
                                                ↓
                                  Asset_Library_Browser_Colab
                                                ↓
                                  Three.js / WebGPU game engine
```

## Requirements
* **GPU:** NVIDIA, ≥ 12 GB VRAM (T4 15 GB works for ≤15 frames; L4/A100 needed for longer clips)
* **RAM:** ≥ 12 GB
* **Disk:** ≈ 10 GB free (PyTorch + CUDA + 2.75 GB MASt3R + 2.29 GB DUSt3R + working space)
* **ffmpeg** (`apt-get install ffmpeg`) — required for video frame extraction and MP4 render
* **Time on first run:** 8-12 min (PyTorch + diff-gaussian-rasterization + simple-knn compile + checkpoints)
* **Time on subsequent runs:** 2-3 min (everything cached in your Drive)
* **Per-scene runtime:** 5-15 min for ≤15 frames, ~1 min per additional 5 frames

## License (full breakdown)

| Component | License | Notes |
|---|---|---|
| `wild-gaussian-splatting` wrapper | Apache-2.0 (per README; no LICENSE file) | All rights reserved by author Danylo Kovalenko (reface.ai). |
| `mast3r` (Naver) | **CC BY-NC-SA 4.0** | Non-commercial, share-alike, attribution required. |
| `dust3r` (Naver) | **CC BY-NC-SA 4.0** | Non-commercial, share-alike, attribution required. |
| `gaussian-splatting` (INRIA) | **INRIA non-commercial research license** | Commercial use requires written consent from Inria. |
| MASt3R checkpoint | CC BY-NC-SA 4.0 | Includes additional dataset-license conditions (see `CHECKPOINTS_NOTICE`). |
| **Your output (.ply, .mp4)** | **Yours** | No restriction. |

**This notebook is for research and evaluation only. Do not use the resulting 3DGS
scenes for commercial purposes without obtaining appropriate licenses from Inria
and Naver.**


In [ ]:
#@title STEP 1 — Install dependencies, clone repo, download checkpoints
"""
• apt-get install ffmpeg (for video frame extraction + MP4 render)
• torch 2.4.1 + cu121 + torchvision 0.19.1 (pinned for upstream compat)
• git clone --recursive nerlfield/wild-gaussian-splatting (3 submodules: dust3r, mast3r, gaussian-splatting)
• pip install top-level + DUSt3R/MASt3R requirements (skips broken `sql` line)
• Build RoPE CUDA kernel (~1-2 min, optional but recommended)
• pip install -e the two 3DGS submodules (diff-gaussian-rasterization + simple-knn, ~3-5 min compile)
• Download MASt3R checkpoint (2.75 GB) + DUSt3R checkpoint (2.29 GB) to Drive cache
All cached at /content/drive/MyDrive/AEI_3D_Cache/WildGS/
"""
import os, sys, time, json, subprocess, shutil, pathlib, traceback

print('='*72)
print('Wild Gaussian Splatting — Install + Setup')
print('='*72)
try:
    import torch
    print(f'  torch        : {torch.__version__}  (CUDA {torch.version.cuda or "none"})')
    print(f'  CUDA avail   : {torch.cuda.is_available()}')
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            p = torch.cuda.get_device_properties(i)
            print(f'  GPU {i}        : {p.name}  ({p.total_memory / (1024**3):.1f} GB)')
    else:
        print('  WARNING: no GPU detected — inference will be very slow on CPU')
except ImportError:
    torch = None
    print('  torch not yet installed (will be installed below)')
print()

CONNECT_GOOGLE_DRIVE = True  #@param {type:'boolean'}
if CONNECT_GOOGLE_DRIVE:
    drive_root = pathlib.Path('/content/drive/MyDrive/AEI_3D_Cache/WildGS')
    drive_root.mkdir(parents=True, exist_ok=True)
    print(f'  Drive cache  : {drive_root}')
    os.environ['HF_HOME'] = str(drive_root / 'huggingface')
    os.environ['HUGGINGFACE_HUB_CACHE'] = str(drive_root / 'huggingface')
else:
    drive_root = pathlib.Path('/content/_wgs_cache')
    drive_root.mkdir(parents=True, exist_ok=True)
    os.environ['HF_HOME'] = str(drive_root / 'huggingface')
    print(f'  Local cache  : {drive_root}  (lost on runtime disconnect)')

os.environ.setdefault('XFORMERS_IGNORE_FLASH_VERSION_CHECK', '1')
REPO_DIR = drive_root / 'wild-gaussian-splatting'
t_total = time.time()

# 1. ffmpeg (apt) ─────────────────────────────────────────────
print('\n[1/7] Installing ffmpeg ...')
r = subprocess.run(['apt-get', 'install', '-y', '-qq', 'ffmpeg'],
                   capture_output=True, text=True)
if r.returncode != 0:
    print('  WARNING: ffmpeg install failed — will use imageio-ffmpeg fallback')
else:
    r = subprocess.run(['ffmpeg', '-version'], capture_output=True, text=True)
    print(f'  ffmpeg: {r.stdout.splitlines()[0] if r.stdout else "installed"}')

# 2. PyTorch ──────────────────────────────────────────────────
if torch is None or not torch.cuda.is_available() or not torch.__version__.startswith('2.4'):
    print('\n[2/7] Installing PyTorch 2.4.1 + cu121 ...')
    t0 = time.time()
    r = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade',
         'torch==2.4.1', 'torchvision==0.19.1', 'torchaudio==2.4.1',
         '--index-url', 'https://download.pytorch.org/whl/cu121'],
        capture_output=True, text=True,
    )
    if r.returncode != 0:
        print('  ERROR:', r.stderr[-1500:])
        raise RuntimeError('PyTorch install failed')
    print(f'  PyTorch installed in {time.time()-t0:.0f}s')

# 3. Clone the upstream repo + submodules ─────────────────────
if not (REPO_DIR / '.git').exists():
    print(f'\n[3/7] Cloning nerlfield/wild-gaussian-splatting into {REPO_DIR} ...')
    t0 = time.time()
    r = subprocess.run(
        ['git', 'clone', '--recurse-submodules',
         'https://github.com/nerlfield/wild-gaussian-splatting.git',
         str(REPO_DIR)],
        capture_output=True, text=True,
    )
    if r.returncode != 0:
        print('  git clone failed:', r.stderr[-500:])
        raise RuntimeError('git clone failed')
    print(f'  cloned (with submodules) in {time.time()-t0:.0f}s')
else:
    print(f'\n[3/7] Reusing cached repo at {REPO_DIR}')
    if not (REPO_DIR / 'mast3r' / '.git').exists():
        print('  initialising submodules ...')
        subprocess.run(['git', 'submodule', 'update', '--init', '--recursive'],
                       cwd=str(REPO_DIR), capture_output=True)

sys.path.insert(0, str(REPO_DIR / 'src'))
sys.path.insert(0, str(REPO_DIR / 'mast3r'))
sys.path.insert(0, str(REPO_DIR / 'gaussian-splatting'))

# 4. Top-level + DUSt3R/MASt3R pip reqs (skip the broken `sql` line)
print('\n[4/7] Installing Python dependencies ...')
t0 = time.time()
req = (REPO_DIR / 'requirements.txt').read_text()
req = chr(10).join(l for l in req.split(chr(10))
                  if l.strip() and not l.strip().startswith('#') and l.strip() != 'sql')
subprocess.check_call(
    [sys.executable, '-m', 'pip', 'install', '--quiet', '--no-build-isolation'] +
    req.split(chr(10)),
    stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT,
)
subprocess.check_call(
    [sys.executable, '-m', 'pip', 'install', '--quiet', '--no-build-isolation',
     '-r', str(REPO_DIR / 'dust3r' / 'requirements.txt')],
    stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT,
)
try:
    subprocess.check_call(
        [sys.executable, '-m', 'pip', 'install', '--quiet', '--no-build-isolation',
         '-r', str(REPO_DIR / 'dust3r' / 'requirements_optional.txt')],
        stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT,
    )
except Exception:
    print('  optional DUSt3R deps install failed (non-fatal)')
print(f'  pip deps installed in {time.time()-t0:.0f}s')

# 5. Build RoPE CUDA kernel (optional, ~1-2 min) ──────────────
rope_dir = REPO_DIR / 'mast3r' / 'dust3r' / 'croco' / 'models' / 'curope'
rope_built = (rope_dir / 'curope').exists() and any(rope_dir.rglob('*.so'))
if rope_built:
    print('\n[5/7] Reusing cached RoPE kernel build')
else:
    print('\n[5/7] Compiling RoPE CUDA kernel (1-2 min) ...')
    t0 = time.time()
    try:
        if not rope_dir.exists():
            print('  curope path not found — skipping (encoder will use Python fallback)')
        else:
            subprocess.check_call(
                [sys.executable, 'setup.py', 'build_ext', '--inplace'],
                cwd=str(rope_dir), stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT,
            )
            print(f'  RoPE kernel built in {time.time()-t0:.0f}s')
    except Exception as e:
        print(f'  RoPE kernel build failed (non-fatal): {e}')

# 6. Build 3DGS submodules (diff-gaussian-rasterization + simple-knn)
print('\n[6/7] Compiling INRIA 3DGS submodules (3-5 min) ...')
t0 = time.time()
for sub in ('diff-gaussian-rasterization', 'simple-knn'):
    sub_path = REPO_DIR / 'gaussian-splatting' / 'submodules' / sub
    if not sub_path.exists():
        print(f'  {sub}: not found, skipping')
        continue
    try:
        r = subprocess.run(
            [sys.executable, '-c', f'import {sub.replace("-", "_")}'],
            capture_output=True, text=True,
        )
        if r.returncode == 0:
            print(f'  {sub}: already installed')
            continue
        subprocess.check_call(
            [sys.executable, '-m', 'pip', 'install', '--quiet', '--no-build-isolation',
             '-e', str(sub_path)],
            stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT,
        )
        print(f'  {sub}: compiled in {time.time()-t0:.0f}s')
    except Exception as e:
        print(f'  {sub}: compile failed: {e}')
print(f'  3DGS submodules done in {time.time()-t0:.0f}s')

# 7. Download MASt3R + DUSt3R checkpoints ──────────────────────
print('\n[7/7] Downloading MASt3R + DUSt3R checkpoints ...')
from tqdm.auto import tqdm
t0 = time.time()
ckpts = [
    ('mast3r', 'https://download.europe.naverlabs.com/ComputerVision/MASt3R/MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric.pth'),
    ('dust3r', 'https://download.europe.naverlabs.com/ComputerVision/DUSt3R/DUSt3R_ViTLarge_BaseDecoder_512_dpt.pth'),
]
import urllib.request
for name, url in ckpts:
    fname = url.split('/')[-1]
    if name == 'mast3r':
        target = REPO_DIR / 'mast3r' / 'checkpoints' / fname
    else:
        target = REPO_DIR / 'dust3r' / 'checkpoints' / fname
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists() and target.stat().st_size > 2_000_000_000:
        print(f'  cached : {fname}  ({target.stat().st_size//(1024*1024)} MB)')
        continue
    print(f'  download : {fname}  (≈2.3-2.8 GB, 2-5 min)')
    with tqdm(unit='B', unit_scale=True, desc=name, miniters=1) as pbar:
        def _hook(n, bs, total):
            pbar.total = total or pbar.total
            pbar.update(bs)
        urllib.request.urlretrieve(url, str(target), reporthook=_hook)
    print(f'    -> {target}  ({target.stat().st_size//(1024*1024)} MB)')
print(f'  checkpoints ready in {time.time()-t0:.0f}s')

print()
print('='*72)
print(f'  STEP 1 complete  (total {time.time()-t_total:.0f}s)')
print('  Next: run STEP 2 (imports + lazy model load)')
print('  Reminder: this notebook is for research use only (see license notice)')
print('='*72)


In [ ]:
#@title STEP 2 — Imports & lazy MASt3R + 3DGS model load
"""
Imports the upstream `mast3r` + `dust3r` + 3DGS packages (one-time cost).
Defines:
  • `run_mast3r(image_dir, output_dir, lr1, niter1, lr2, niter2, ...)` —
    runs MASt3R's `sparse_global_alignment`, exports COLMAP-format files
  • `run_3dgs(colmap_dir, output_dir, iterations, ...)` — trains the
    INRIA 3DGS scene as a subprocess (all lr / densify / antialiasing /
    sh_degree params exposed)
  • `run_full_pipeline(...)` — chains both with a single call
  • `_render_orbit(...)` — uses the upstream's
    `generate_ellipse_path_from_camera_infos` to interpolate an orbit
    through the training-view centroid
All MASt3R + 3DGS params match the upstream defaults exactly.
"""
import sys, os, time, json, pathlib, warnings, shutil
warnings.filterwarnings('ignore')

WGS_DIR = pathlib.Path(os.environ.get('AEI_WGS_REPO',
                    str(pathlib.Path('/content/drive/MyDrive/AEI_3D_Cache/WildGS/wild-gaussian-splatting'))))
for p in [WGS_DIR / 'src', WGS_DIR / 'mast3r', WGS_DIR / 'gaussian-splatting']:
    sp = str(p)
    if sp not in sys.path:
        sys.path.insert(0, sp)
print(f'  repo        : {WGS_DIR}')
print(f'  src on path : {(WGS_DIR / "src").exists()}')
print(f'  mast3r      : {(WGS_DIR / "mast3r").exists()}')
print(f'  gs          : {(WGS_DIR / "gaussian-splatting").exists()}')
print()

import torch
import numpy as np
from PIL import Image
from tqdm.auto import tqdm

print('  Loading upstream modules (one-time cost) ...')
t0 = time.time()
try:
    from mast3r.model import load_model
    print(f'  mast3r.model              ✓')
except Exception as e:
    print(f'  mast3r.model              ✗  {type(e).__name__}: {e}')
    raise
from mast3r.cloud_opt.sparse_ga import sparse_global_alignment
print(f'  mast3r.cloud_opt.sparse_ga ✓')
from dust3r.utils.image import load_images
print(f'  dust3r.utils.image         ✓')
from dust3r.utils.device import to_numpy
print(f'  dust3r.utils.device        ✓')
from dust3r.image_pairs import make_pairs
print(f'  dust3r.image_pairs         ✓')
from src.colmap_dataset_utils import storePly
print(f'  src.colmap_dataset_utils   ✓')
print(f'  total import: {time.time()-t0:.1f}s')
print()

MAST3R_CKPT = str(WGS_DIR / 'mast3r' / 'checkpoints' /
                 'MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric.pth')

# ── MASt3R model cache ──────────────────────────────────────
_MODEL_CACHE = {}
def get_mast3r_model(ckpt_path=MAST3R_CKPT, device='cuda'):
    if ckpt_path in _MODEL_CACHE:
        return _MODEL_CACHE[ckpt_path]
    if not pathlib.Path(ckpt_path).exists():
        raise FileNotFoundError(
            f'MASt3R checkpoint not found: {ckpt_path}. '
            'Re-run STEP 1 to download.'
        )
    print(f'  Loading MASt3R from {ckpt_path} ...')
    t0 = time.time()
    model = load_model(ckpt_path, device=device)
    _MODEL_CACHE[ckpt_path] = model
    print(f'  model loaded in {time.time()-t0:.1f}s')
    return model

# ── MASt3R: image-folder -> COLMAP-format dataset ──────────
def _image_files_in(folder, max_n=32):
    """Find images in a folder (sorted), cap at max_n for VRAM safety."""
    SUPP = {'.png', '.jpg', '.jpeg', '.JPG', '.PNG'}
    files = sorted(
        p for p in pathlib.Path(folder).rglob('*')
        if p.suffix.lower() in {s.lower() for s in SUPP}
    )
    if not files:
        raise SystemExit(f'No images found in {folder}')
    if len(files) > max_n:
        print(f'  WARN: {len(files)} images found; using first {max_n} (T4 VRAM limit)')
        files = files[:max_n]
    return files

def run_mast3r(
    image_dir, output_dir,
    image_size=512,
    lr1=0.07, niter1=300, lr2=0.01, niter2=300,
    matching_conf_thr=5.0, shared_intrinsics=True,
    opt_pp=True, opt_depth=True, loss_dust3r_w=0.01,
    subsample=8, kinematic_mode='hclust-ward',
    device='cuda',
):
    """Run MASt3R pose + point cloud estimation. Exports COLMAP-format dataset.

    All args match the upstream `sparse_scene_optimizer` defaults.
    See https://github.com/naver/mast3r/blob/main/mast3r/cloud_opt/sparse_ga.py
    for the full param list."""
    image_dir = pathlib.Path(image_dir).resolve()
    output_dir = pathlib.Path(output_dir).resolve()
    output_dir.mkdir(parents=True, exist_ok=True)
    files = _image_files_in(image_dir)
    print(f'  MASt3R on {len(files)} images @ {image_size}px ...')
    model = get_mast3r_model(device=device)
    imgs = load_images([str(f) for f in files], size=image_size, square_ok=True, verbose=True)
    print(f'  images loaded: {len(imgs)}')
    pairs = make_pairs(imgs, scene_graph='complete', prefilter=None, symmetrize=True)
    print(f'  pairs: {len(pairs)}')
    cache_dir = output_dir / 'mast3r_cache'
    cache_dir.mkdir(parents=True, exist_ok=True)
    print(f'  sparse global alignment (kinematic={kinematic_mode}, subsample={subsample}) ...')
    t0 = time.time()
    scene = sparse_global_alignment(
        [str(f) for f in files], pairs, str(cache_dir), model,
        subsample=subsample, kinematic_mode=kinematic_mode,
        device=device, shared_intrinsics=shared_intrinsics,
        lr1=lr1, niter1=niter1, lr2=lr2, niter2=niter2,
        opt_pp=opt_pp, opt_depth=opt_depth,
        matching_conf_thr=matching_conf_thr, loss_dust3r_w=loss_dust3r_w,
    )
    elapsed = time.time() - t0
    print(f'  alignment done in {elapsed:.1f}s')
    # Export COLMAP-format dataset
    colmap_dir = output_dir / 'colmap'
    colmap_dir.mkdir(parents=True, exist_ok=True)
    (colmap_dir / 'images').mkdir(exist_ok=True)
    (colmap_dir / 'masks').mkdir(exist_ok=True)
    (colmap_dir / 'sparse' / '0').mkdir(parents=True, exist_ok=True)
    poses    = scene.get_im_poses().detach().cpu().numpy()
    pps      = scene.get_principal_points().detach().cpu().numpy()
    focals   = scene.get_focals().detach().cpu().numpy()
    pts3d    = scene.get_dense_pts3d(clean_depth=False)
    import cv2
    for i, f in enumerate(files):
        img = cv2.imread(str(f))
        cv2.imwrite(str(colmap_dir / 'images' / f'{i}.png'), img)
        h, w = img.shape[:2]
        mask = np.ones((h, w), dtype=np.uint8) * 255
        cv2.imwrite(str(colmap_dir / 'masks' / f'{i}.png'), mask)
    cams = []
    for i in range(len(files)):
        cams.append(f'{i+1} PINHOLE {focals[i,0]:.6f} {focals[i,1]:.6f} {pps[i,0]:.6f} {pps[i,1]:.6f}\n')
    (colmap_dir / 'sparse' / '0' / 'cameras.txt').write_text(''.join(cams))
    imgs_text = []
    for i in range(len(files)):
        c2w = poses[i]
        R = c2w[:3, :3]
        t = c2w[:3, 3]
        from scipy.spatial.transform import Rotation
        quat = Rotation.from_matrix(R).as_quat()  # [x, y, z, w]
        qx, qy, qz, qw = quat
        imgs_text.append(
            f'{i+1} {qw:.6f} {qx:.6f} {qy:.6f} {qz:.6f} '
            f'{t[0]:.6f} {t[1]:.6f} {t[2]:.6f} {i+1} {i}.png\n\n'
        )
    (colmap_dir / 'sparse' / '0' / 'images.txt').write_text(''.join(imgs_text))
    pts = pts3d[0].reshape(-1, 3) if hasattr(pts3d, 'reshape') else np.asarray(pts3d[0]).reshape(-1, 3)
    n_pts = pts.shape[0]
    ply_lines = [
        'ply\nformat ascii 1.0\n',
        f'element vertex {n_pts}\n',
        'property float x\nproperty float y\nproperty float z\n',
        'end_header\n',
    ]
    for p in pts:
        ply_lines.append(f'{p[0]:.4f} {p[1]:.4f} {p[2]:.4f}\n')
    (colmap_dir / 'sparse' / '0' / 'points3D.ply').write_text(''.join(ply_lines))
    print(f'  COLMAP dataset written to {colmap_dir}')
    return colmap_dir, len(files)

# ── 3DGS training (subprocess, since the upstream uses an old API) ────
def run_3dgs(
    colmap_dir, output_dir,
    iterations=7000,
    position_lr_init=1.6e-4, position_lr_final=1.6e-6,
    feature_lr=2.5e-3, opacity_lr=0.05, scaling_lr=5e-3, rotation_lr=1e-3,
    densify_from_iter=500, densify_until_iter=15000,
    densification_interval=100, opacity_reset_interval=3000,
    densify_grad_threshold=2e-4, lambda_dssim=0.2, percent_dense=0.01,
    sh_degree=3, antialiasing=False, white_background=False,
    data_device='cuda', render_output=True, num_render_frames=120,
):
    """Train INRIA 3DGS on the COLMAP dataset, then render an orbit MP4."""
    colmap_dir = pathlib.Path(colmap_dir).resolve()
    output_dir = pathlib.Path(output_dir).resolve()
    output_dir.mkdir(parents=True, exist_ok=True)
    gs_root = WGS_DIR / 'gaussian-splatting'
    if str(gs_root) not in sys.path:
        sys.path.insert(0, str(gs_root))
    cmd = [
        sys.executable, 'train.py', '-s', str(colmap_dir), '-m', str(output_dir),
        '--iterations', str(iterations),
        '--position_lr_init', str(position_lr_init),
        '--position_lr_final', str(position_lr_final),
        '--position_lr_delay_mult', '0.01',
        '--position_lr_max_steps', str(iterations),
        '--feature_lr', str(feature_lr),
        '--opacity_lr', str(opacity_lr),
        '--scaling_lr', str(scaling_lr),
        '--rotation_lr', str(rotation_lr),
        '--percent_dense', str(percent_dense),
        '--lambda_dssim', str(lambda_dssim),
        '--densify_from_iter', str(densify_from_iter),
        '--densify_until_iter', str(densify_until_iter),
        '--densification_interval', str(densification_interval),
        '--opacity_reset_interval', str(opacity_reset_interval),
        '--densify_grad_threshold', str(densify_grad_threshold),
        '--sh_degree', str(sh_degree),
        '--save_iterations', str(iterations),
        '--test_iterations', str(iterations),
        '--checkpoint_iterations', str(iterations),
        '--data_device', data_device,
    ]
    if antialiasing:
        cmd.append('--antialiasing')
    if white_background:
        cmd.append('--white_background')
    print(f'  running 3DGS training ({iterations} iters, sh_degree={sh_degree}, '
          f'antialiasing={antialiasing}, white_bg={white_background}) ...')
    t0 = time.time()
    r = subprocess.run(
        cmd, cwd=str(gs_root), capture_output=True, text=True,
    )
    if r.returncode != 0:
        print('  3DGS training failed:')
        print(r.stderr[-2000:])
        raise RuntimeError('3DGS training failed')
    print(f'  3DGS training done in {time.time()-t0:.0f}s')
    ply = output_dir / 'point_cloud' / f'iteration_{iterations}' / 'point_cloud.ply'
    if not ply.exists():
        raise FileNotFoundError(f'3DGS output not found: {ply}')
    if render_output:
        _render_orbit(colmap_dir, ply, output_dir, num_render_frames,
                       sh_degree=sh_degree, white_background=white_background)
    return ply

def _render_orbit(colmap_dir, ply_path, gs_dir, num_frames=120,
                  sh_degree=3, white_background=False):
    """Render an orbit MP4 using the upstream's generate_ellipse_path_from_camera_infos.
    The function is copied inline from the nerlfield/gaussian-splatting fork."""
    from utils.graphics_utils import getWorld2View2
    from utils.camera_utils import generate_ellipse_path_from_camera_infos
    from scene.cameras import Camera
    from utils.graphics_utils import focal2fov, fov2focal, getProjectionMatrix
    from gaussian_renderer import render
    from scene.gaussian_model import GaussianModel
    import torchvision
    colmap_dir = pathlib.Path(colmap_dir).resolve()
    gs_dir = pathlib.Path(gs_dir).resolve()
    render_dir = gs_dir / 'render_orbit'
    render_dir.mkdir(parents=True, exist_ok=True)
    renders_imgs_dir = render_dir / 'frames'
    renders_imgs_dir.mkdir(parents=True, exist_ok=True)
    # Load the existing images.txt as input to generate_ellipse_path_from_camera_infos
    # by constructing a synthetic CameraInfo list
    sys.path.insert(0, str(WGS_DIR / 'gaussian-splatting'))
    sys.path.insert(0, str(WGS_DIR / 'gaussian-splatting' / 'utils'))
    # Parse the COLMAP images.txt we wrote
    images_txt = (colmap_dir / 'sparse' / '0' / 'images.txt').read_text().strip()
    cam_lines = [l for l in images_txt.split('\n') if l and not l.startswith('#')]
    pose_lines = cam_lines[::2]  # image lines (empty lines interleaved)
    cams_txt = (colmap_dir / 'sparse' / '0' / 'cameras.txt').read_text().strip()
    cam_specs = {}
    for l in cams_txt.split('\n'):
        if l.startswith('#') or not l.strip():
            continue
        parts = l.split()
        if parts[1] == 'PINHOLE':
            cam_id = int(parts[0])
            fx, fy, cx, cy = (float(x) for x in parts[2:6])
            cam_specs[cam_id] = (fx, fy, cx, cy)
    # Read image dimensions
    first_img = cv2.imread(str(colmap_dir / 'images' / '0.png'))
    H, W = first_img.shape[:2]
    # Build CameraInfo list
    from utils.camera_utils import CameraInfo
    cam_infos = []
    for i, line in enumerate(pose_lines):
        if not line.strip():
            continue
        parts = line.split()
        cam_id = int(parts[0])
        qw, qx, qy, qz = (float(x) for x in parts[1:5])
        tx, ty, tz = (float(x) for x in parts[5:8])
        # COLMAP convention: world-to-camera rotation = R^T of qvec, translation = (tx,ty,tz)
        from scipy.spatial.transform import Rotation
        R_w2c = Rotation.from_quat([qx, qy, qz, qw]).as_matrix()
        T_w2c = np.array([tx, ty, tz])
        R = R_w2c.T  # CameraInfo stores R transposed
        T = T_w2c
        fx, fy, cx, cy = cam_specs[cam_id]
        from utils.graphics_utils import fov2focal, focal2fov
        FovY = focal2fov(fy, H)
        FovX = focal2fov(fx, W)
        img = first_img  # placeholder image (not actually used for rendering)
        cam_infos.append(CameraInfo(uid=i, R=R, T=T, FovY=FovY, FovX=FovX,
                                     image=img, image_path='', image_name=f'{i:05d}.png',
                                     width=W, height=H))
    print(f'  generating orbit with {len(cam_infos)} input cams, {num_frames} frames')
    render_cam_infos = generate_ellipse_path_from_camera_infos(cam_infos, n_frames=num_frames)
    # Load the trained GaussianModel
    ply = pathlib.Path(ply_path).resolve()
    gaussians = GaussianModel(sh_degree)
    gaussians.load_ply(str(ply))
    bg = torch.tensor([1, 1, 1] if white_background else [0, 0, 0],
                      dtype=torch.float32, device='cuda')
    print(f'  rendering {len(render_cam_infos)} orbit frames ...')
    with torch.no_grad():
        for i, cam_info in enumerate(tqdm(render_cam_infos, desc='orbit')):
            # Build a Camera object compatible with the INRIA 3DGS renderer
            R_w2c = cam_info.R.T  # un-transpose (we stored R transposed)
            T_w2c = cam_info.T
            image = torch.zeros((3, H, W), device='cuda')  # dummy image (not used)
            FovX = cam_info.FovX
            FovY = cam_info.FovY
            camera = Camera(colmap_id=i, R=R_w2c, T=T_w2c, FoVx=FovX, FoVy=FovY,
                           image=image, gt_alpha_mask=None, image_name=cam_info.image_name,
                           uid=i, data_device='cuda')
            from argparse import Namespace
            pipe = Namespace(convert_SHs_python=False, compute_cov3D_python=False,
                             debug=False, antialiasing=False)
            rendering = render(camera, gaussians, pipe, bg)['render']
            torchvision.utils.save_image(
                rendering,
                str(renders_imgs_dir / f'{i:05d}.png')
            )
    # Encode the orbit as MP4
    mp4_path = render_dir / 'orbit.mp4'
    cmd = ['ffmpeg', '-y', '-framerate', '24',
           '-i', str(renders_imgs_dir / '%05d.png'),
           '-vf', 'pad=ceil(iw/2)*2:ceil(ih/2)*2',
           '-c:v', 'libx264', '-pix_fmt', 'yuv420p', '-crf', '23',
           str(mp4_path)]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(f'  ffmpeg MP4 encode failed: {r.stderr[-500:]}')
        return renders_imgs_dir
    print(f'  orbit MP4: {mp4_path}  ({mp4_path.stat().st_size//(1024*1024)} MB)')
    return mp4_path

# ── Full pipeline (MASt3R + 3DGS) ────────────────────────────
def run_full_pipeline(
    image_dir, output_dir,
    image_size=512, iterations=7000, do_drive_save=True, **kwargs,
):
    """MASt3R alignment + 3DGS training. Returns the final .ply path."""
    image_dir = pathlib.Path(image_dir).resolve()
    output_dir = pathlib.Path(output_dir).resolve()
    output_dir.mkdir(parents=True, exist_ok=True)
    colmap_dir, n_imgs = run_mast3r(image_dir, output_dir,
                                    image_size=image_size, **kwargs)
    ply = run_3dgs(colmap_dir, output_dir, iterations=iterations)
    if do_drive_save:
        _mirror_to_drive([ply], str(output_dir))
    return ply

def _mirror_to_drive(paths, output_root, drive_subdir='WildGS'):
    if not paths:
        return
    drive_base = pathlib.Path('/content/drive/MyDrive/AEI_3D_Out') / drive_subdir
    try:
        drive_base.parent.mkdir(parents=True, exist_ok=True)
        for src in paths:
            try:
                _src = pathlib.Path(src) if not isinstance(src, pathlib.Path) else src
                dst = drive_base / _src.relative_to(pathlib.Path(output_root))
                dst.parent.mkdir(parents=True, exist_ok=True)
                if _src.resolve() == dst.resolve():
                    continue
                if dst.exists() and dst.stat().st_size == _src.stat().st_size:
                    continue
                shutil.copy2(str(_src), str(dst))
            except Exception as e:
                print(f'  WARN: drive mirror of {_src.name} failed: {e}')
        print(f'  drive mirror: {drive_base}')
    except Exception as e:
        print(f'  drive mirror skipped: {e}')

print('  pipeline ready: run_mast3r, run_3dgs, run_full_pipeline, _mirror_to_drive')


In [ ]:
#@title STEP 3 — Core helpers (extract frames, run pipeline, batch)
"""
Re-exports the pipeline from STEP 2 and adds:
  • `extract_video_frames(video_path, output_dir, fps=2, max_frames=32)`
    — ffmpeg-based video frame extraction
  • `run_single_scene(image_or_video, output_dir, ...)` — convenience
  • `run_batch(input_dir, output_dir, ...)` — process multiple subdirs
"""
import subprocess, json

def extract_video_frames(video_path, output_dir, fps=2, max_frames=32):
    """Use ffmpeg to extract `fps` frames per second from a video, capped at max_frames."""
    output_dir = pathlib.Path(output_dir).resolve()
    output_dir.mkdir(parents=True, exist_ok=True)
    pattern = output_dir / '%d.png'
    cmd = [
        'ffmpeg', '-y', '-i', str(video_path),
        '-vf', f'fps={fps},scale=iw:ih',
        '-frames:v', str(max_frames),
        str(pattern),
    ]
    print(f'  extracting ≤{max_frames} frames @ {fps} fps from {pathlib.Path(video_path).name} ...')
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print('  ffmpeg failed:', r.stderr[-500:])
        raise RuntimeError('ffmpeg frame extraction failed')
    n = len(list(output_dir.glob('*.png')))
    print(f'  extracted {n} frames to {output_dir}')
    return output_dir, n

def run_single_scene(image_or_video, output_dir,
                     image_size=512, iterations=7000,
                     video_fps=2, do_drive_save=True, **kwargs):
    """Run the full MASt3R + 3DGS pipeline on one image folder OR one video.
    Auto-detects which by file extension."""
    src = pathlib.Path(image_or_video).resolve()
    out = pathlib.Path(output_dir).resolve()
    out.mkdir(parents=True, exist_ok=True)
    if src.suffix.lower() in {'.mp4', '.mov', '.webm', '.avi', '.mkv', '.m4v'}:
        frames_dir = out / 'frames'
        frames_dir.mkdir(parents=True, exist_ok=True)
        if not any(frames_dir.glob('*.png')):
            extract_video_frames(src, frames_dir, fps=video_fps)
        image_dir = frames_dir
    else:
        image_dir = src if src.is_dir() else src.parent
    ply = run_full_pipeline(
        image_dir, out, image_size=image_size, iterations=iterations,
        do_drive_save=do_drive_save, **kwargs,
    )
    return ply

def run_batch(input_dir, output_dir,
              image_size=512, iterations=7000, video_fps=2,
              do_drive_save=True, **kwargs):
    """Process every subdirectory of `input_dir` (or every video file at top level)
    as a separate scene. Already-done outputs are skipped."""
    in_dir = pathlib.Path(input_dir).resolve()
    if not in_dir.exists():
        raise SystemExit(f'Input not found: {in_dir}')
    out_dir = pathlib.Path(output_dir).resolve()
    out_dir.mkdir(parents=True, exist_ok=True)
    SUPP = {'.mp4', '.mov', '.webm', '.avi', '.mkv', '.m4v'}
    videos = sorted(p for p in in_dir.glob('*') if p.suffix.lower() in SUPP)
    subdirs = sorted(p for p in in_dir.iterdir() if p.is_dir() and any(p.iterdir()))
    scenes = [(p, p.stem) for p in videos] + [(p, p.name) for p in subdirs]
    if not scenes:
        raise SystemExit(f'No videos or subdirs found in {in_dir}')
    n_ok = 0
    for src, slug in scenes:
        scene_out = out_dir / slug
        scene_out.mkdir(parents=True, exist_ok=True)
        ply = scene_out / 'point_cloud' / f'iteration_{iterations}' / 'point_cloud.ply'
        if ply.exists() and ply.stat().st_size > 1024:
            print(f'  skip: {slug}  (already done)')
            continue
        try:
            run_single_scene(src, scene_out,
                             image_size=image_size, iterations=iterations,
                             video_fps=video_fps, do_drive_save=False, **kwargs)
            n_ok += 1
        except Exception as e:
            print(f'  ERROR on {slug}: {type(e).__name__}: {e}')
    if do_drive_save and n_ok > 0:
        all_plys = sorted(out_dir.rglob(f'iteration_{iterations}/point_cloud.ply'))
        if all_plys:
            _mirror_to_drive(all_plys, str(out_dir))
    print(f'\n  done: {n_ok} new scene(s) processed')
    return n_ok

print('  scene helpers ready: extract_video_frames, run_single_scene, run_batch')


In [ ]:
#@title STEP 4 — Launch Gradio UI
"""
Interactive Gradio UI:
  • Video upload (gr.Video) OR image folder upload (gr.File multiple)
  • MASt3R params: image size, lr1/niter1 (coarse), lr2/niter2 (refinement),
    matching_conf_thr, opt_pp, opt_depth, kinematic_mode
  • 3DGS params: iterations, sh_degree, antialiasing, white_background,
    opacity_reset_interval
  • Video frame-rate slider (1-5 fps extracted)
  • Run button → .ply + .mp4 outputs
  • License notice at the top of the UI
"""
import os, sys, time, json, shutil, pathlib, traceback, tempfile
import gradio as gr

gr.TEMP_DIR = '/tmp/gradio_wgs'
os.makedirs(gr.TEMP_DIR, exist_ok=True)

def _do_run(video_file, files, image_size, iterations, video_fps,
            lr1, niter1, lr2, niter2, matching_conf_thr,
            opt_pp, opt_depth, kinematic_mode,
            sh_degree, antialiasing, white_background,
            opacity_reset_interval, num_orbit_frames,
            do_drive_save, progress=gr.Progress()):
    if not video_file and not files:
        return 'Please upload a video or a folder of images.', None, None, None
    try:
        tmp = pathlib.Path(tempfile.mkdtemp(prefix='wgs_gradio_'))
        if video_file:
            src = pathlib.Path(video_file if isinstance(video_file, str) else video_file.name)
            progress(0.05, 'Extracting video frames...')
            frames_dir, n_frames = extract_video_frames(src, tmp / 'frames', fps=int(video_fps))
        else:
            frames_dir = tmp / 'frames'
            frames_dir.mkdir(parents=True, exist_ok=True)
            for f in files:
                src = pathlib.Path(f.name if hasattr(f, 'name') else f)
                shutil.copy2(str(src), str(frames_dir / src.name))
            n_frames = len(list(frames_dir.iterdir()))
        progress(0.15, f'Running MASt3R on {n_frames} frames ...')
        colmap_dir, n = run_mast3r(
            str(frames_dir), tmp / 'scene',
            image_size=int(image_size),
            lr1=lr1, niter1=int(niter1), lr2=lr2, niter2=int(niter2),
            matching_conf_thr=matching_conf_thr,
            opt_pp=opt_pp, opt_depth=opt_depth,
            kinematic_mode=kinematic_mode,
        )
        progress(0.55, f'Training 3DGS ({iterations} iterations, this is the long step) ...')
        ply = run_3dgs(
            str(colmap_dir), tmp / 'scene' / 'gs',
            iterations=int(iterations),
            sh_degree=int(sh_degree),
            antialiasing=antialiasing,
            white_background=white_background,
            opacity_reset_interval=int(opacity_reset_interval),
            num_render_frames=int(num_orbit_frames),
        )
        progress(0.95, 'Rendering orbit MP4 ...')
        mp4 = tmp / 'scene' / 'gs' / 'render_orbit' / 'orbit.mp4'
        if not mp4.exists():
            mp4 = tmp / 'scene' / 'gs' / 'render_orbit' / 'frames'
        if do_drive_save:
            outputs = [p for p in (ply, mp4 if mp4.exists() else None) if p is not None]
            _mirror_to_drive(outputs, str(tmp / 'scene' / 'gs'))
        progress(1.0, 'Done')
        return (f'Generated 3DGS scene from {n} frames → '
                f'{ply.name}  ({ply.stat().st_size//(1024*1024)} MB)'), \
               str(ply), str(mp4) if mp4.exists() else None, \
               _splat_viewer_html(str(ply))
    except Exception as e:
        traceback.print_exc()
        return f'Error: {type(e).__name__}: {e}', None, None, None

def _splat_viewer_html(ply_path):
    if not ply_path or not pathlib.Path(ply_path).exists():
        return '<p style="color:#888">No scene to preview</p>'
    p = pathlib.Path(ply_path)
    return (f'<div style="width:100%; height:200px; background:#1a1a1a; display:flex; '
            f'align-items:center; justify-content:center; color:#aaa; font-family:monospace; '
            f'padding:20px; text-align:center; border-radius:8px">'
            f'3DGS scene ready: {p.name}  ({p.stat().st_size//(1024*1024)} MB)  '
            f'<br><small>Download the .ply below and open in '
            f'<a href="https://supersplat.dev" target="_blank" style="color:#4af">supersplat.dev</a>, '
            f'<a href="https://playcanvas.com/viewer" target="_blank" style="color:#4af">playcanvas.com/viewer</a>, '
            f'or any 3DGS viewer.</small>'
            f'</div>')

with gr.Blocks(title='Wild Gaussian Splatting · Video → 3DGS (AEI Colab)') as demo:
    gr.Markdown(
        '''
        ## 🎥 [Wild Gaussian Splatting](https://github.com/nerlfield/wild-gaussian-splatting) — Video → 3DGS
        Upload a casual video or a folder of overlapping photos, get a polished 3DGS
        scene back in 5-15 minutes. The closest open-source equivalent to a Luma-style
        capture workflow.
        * **Method:** [MASt3R](https://github.com/naver/mast3r) (Naver, CC BY-NC-SA) for pose +
          point cloud estimation → [INRIA 3DGS](https://github.com/graphdeco-inria/gaussian-splatting)
          (INRIA non-commercial) for per-scene training.
        '''
    )
    gr.Markdown(
        '> ⚠️ **Research / evaluation use only.** The INRIA 3DGS rasterizer and the MASt3R backbone '
        'are non-commercial. Do not use the resulting 3DGS scenes commercially without obtaining '
        'appropriate licenses from Inria and Naver. See the **License** section in the notebook header.'
    )
    with gr.Row():
        with gr.Column(scale=1):
            video_file = gr.Video(
                label='Video  ( .mp4 / .mov / .webm )',
                sources=['upload'],
            )
            files = gr.File(
                label='…or upload a folder of images  ( .jpg / .png )',
                file_count='multiple',
                file_types=['.png', '.jpg', '.jpeg'],
            )
            with gr.Accordion('⚙️ MASt3R + 3DGS Settings', open=False):
                image_size = gr.Slider(
                    256, 512, value=512, step=128,
                    label='MASt3R image size (long side)',
                    info='MASt3R resizes so the long side equals this. 512 = SOTA quality, 256 = faster.'
                )
                with gr.Accordion('MASt3R alignment', open=False):
                    lr1 = gr.Slider(
                        1e-4, 0.5, value=0.07, step=1e-3,
                        label='lr1 (coarse GA learning rate)',
                        info='Learning rate for coarse global alignment (3D matching).'
                    )
                    niter1 = gr.Slider(
                        0, 1000, value=300, step=50,
                        label='niter1 (coarse GA iterations)',
                        info='Iterations for coarse global alignment. 300 is the upstream default.'
                    )
                    lr2 = gr.Slider(
                        1e-4, 0.5, value=0.01, step=1e-3,
                        label='lr2 (refinement learning rate)',
                        info='Learning rate for 2D-reproj refinement. 0 = skip refinement.'
                    )
                    niter2 = gr.Slider(
                        0, 1000, value=300, step=50,
                        label='niter2 (refinement iterations)',
                        info='Iterations for 2D-reproj refinement. 0 = skip refinement.'
                    )
                    matching_conf_thr = gr.Slider(
                        0.1, 20.0, value=5.0, step=0.1,
                        label='matching_conf_thr',
                        info='Pairs below this confidence fall back to DUSt3R loss. Lower = more pairs kept.'
                    )
                    opt_pp = gr.Checkbox(
                        True,
                        label='opt_pp (refine principal point)',
                        info='Optimize the principal point during refinement.'
                    )
                    opt_depth = gr.Checkbox(
                        True,
                        label='opt_depth (refine per-pixel depth)',
                        info='Optimize the per-pixel depth during refinement.'
                    )
                    kinematic_mode = gr.Dropdown(
                        choices=['hclust-ward', 'hclust-complete', 'hclust-average', 'hclust-single', 'mst'],
                        value='hclust-ward',
                        label='Kinematic mode (scene graph)',
                        info="How the input views are chained together. hclust-* use hierarchical clustering (default); mst uses a minimum spanning tree."
                    )
                with gr.Accordion('3DGS training', open=False):
                    iterations = gr.Slider(
                        3000, 30000, value=7000, step=1000,
                        label='3DGS training iterations',
                        info='More iterations = better quality but slower. 7000 is a good balance.'
                    )
                    sh_degree = gr.Dropdown(
                        choices=[0, 1, 2, 3],
                        value=3,
                        label='SH degree (view-dependent shading)',
                        info='0 = flat color, 3 = full view-dependent. Higher = better quality but more memory.'
                    )
                    antialiasing = gr.Checkbox(
                        False,
                        label='antialiasing (Mip-NeRF 360 filter)',
                        info='Enable the antialiasing filter for smoother rendering. Adds ~10% training time.'
                    )
                    white_background = gr.Checkbox(
                        False,
                        label='white_background',
                        info='Treat COLMAP background as white instead of black. Useful for object captures on white backdrops.'
                    )
                    opacity_reset_interval = gr.Slider(
                        1000, 10000, value=3000, step=500,
                        label='opacity_reset_interval',
                        info='Reset all opacities every N iterations. Prevents stuck-floating Gaussians.'
                    )
                    num_orbit_frames = gr.Slider(
                        30, 240, value=120, step=10,
                        label='Orbit MP4 frames',
                        info='Number of frames in the rendered orbit MP4. More = smoother video but slower render.'
                    )
                video_fps = gr.Slider(
                    1, 5, value=2, step=1,
                    label='Video frame rate (extracted)',
                    info='Frames per second to extract from the video. 2 fps is a good default.'
                )
                do_drive_save = gr.Checkbox(
                    True,
                    label='Mirror .ply + .mp4 to Google Drive',
                    info='Copy outputs to /content/drive/MyDrive/AEI_3D_Out/WildGS/'
                )
            run_btn = gr.Button('🚀 Run (5-15 min)', variant='primary')
        with gr.Column(scale=1):
            log = gr.Textbox(label='Status', lines=2, interactive=False)
            ply_out = gr.File(label='Generated 3DGS .ply', interactive=False)
            mp4_out = gr.Video(label='Orbit MP4 preview', interactive=False)
            gr.Markdown(
                '''
                **Next steps for the .ply**
                * Open in [supersplat.dev](https://supersplat.dev) for cleanup + editing
                * Pipe into **[SplatTransform_Colab](https://github.com/Skquark/AEI-Colab-Notebooks/blob/main/SplatTransform_Colab.ipynb)** STEP 3 to compress to game-ready SOG/SPLAT/SPZ/GLB
                * Or load directly in [Three.js / WebGPU](https://github.com/Skquark/AEI-Colab-Notebooks#game-engine-integration--loading-the-assets-into-threejs--webgpu)
                '''
            )
    run_btn.click(
        _do_run,
        inputs=[video_file, files, image_size, iterations, video_fps,
                lr1, niter1, lr2, niter2, matching_conf_thr,
                opt_pp, opt_depth, kinematic_mode,
                sh_degree, antialiasing, white_background,
                opacity_reset_interval, num_orbit_frames,
                do_drive_save],
        outputs=[log, ply_out, mp4_out],
    )
    def _welcome():
        return ('Ready. Upload a video (recommended 5-30 s clip) or a folder of '
                'overlapping photos and click Run.')
    demo.load(_welcome, outputs=[log])

from IPython.display import clear_output
clear_output()
demo.queue(default_concurrency_limit=2).launch(
    share=False, prevent_thread_lock=True, inline=True,
    show_error=True, height=900,
)
print('\n  Gradio UI running above ↑  (cell stays alive — do not re-run)')


In [ ]:
#@title STEP 5 — Keep alive + session summary
"""
Keeps the runtime alive for 12 h. Prints a session summary.
"""
import datetime, time, json, sys, pathlib, os
summary = {
    'timestamp'   : datetime.datetime.utcnow().isoformat() + 'Z',
    'python'      : sys.version.split()[0],
    'torch'       : None, 'cuda': None, 'gpus': [],
    'repo'        : str(WGS_DIR),
    'ffmpeg'      : shutil.which('ffmpeg') is not None,
    'model_cached': list(_MODEL_CACHE.keys()),
    'drive_cache' : str(pathlib.Path('/content/drive/MyDrive/AEI_3D_Cache/WildGS')),
}
try:
    import torch
    summary['torch'] = torch.__version__
    summary['cuda']  = torch.version.cuda
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            p = torch.cuda.get_device_properties(i)
            summary['gpus'].append(f'{p.name}  {p.total_memory // (1024**3)} GB')
except Exception as e:
    summary['torch_error'] = str(e)
print(json.dumps(summary, indent=2))
print()
print('  • STEP 4 is the Gradio UI (interactive video/folder upload)')
print('  • STEP 6 is the Colab single-scene picker')
print('  • STEP 7 is the Colab batch (multiple scenes)')
print('  • Outputs land in OUTPUT_DIR and (if mirrored) in /content/drive/MyDrive/AEI_3D_Out/WildGS/')
print()
print('  ⚠️  REMINDER: This notebook is for research / evaluation only.')
print('      The 3DGS rasterizer is INRIA non-commercial; the MASt3R backbone is CC BY-NC-SA 4.0.')
print()
print('  Cell will keep the runtime alive for 12 h unless you disconnect.')
print('  Press the stop button in the toolbar to release the GPU.')

_start = time.time()
while time.time() - _start < 43200:  # 12 h
    time.sleep(300)
    print(f'  [{int(time.time()-_start)//60} min] still running — close tab to stop')


In [ ]:
#@title STEP 6 — Quick test (Colab single-scene picker)
"""
Quick-test one video OR one image folder. Use this for verification
or for scripting (no Gradio UI). For interactive UI, use STEP 4.
"""
import time, pathlib

INPUT_PATH     = ''  #@param {type:'string', placeholder: '/content/my_video.mp4 or /content/my_photos/'}  # info='Either a video file or a folder of images.'
OUTPUT_DIR     = '/content/WildGS_Out'  #@param {type:'string'}  # info='Where the .ply and .mp4 will be written.'
CKPT_PATH      = ''  #@param {type:'string', placeholder: 'leave empty for default'}  # info='Override the MASt3R checkpoint path.'

IMAGE_SIZE     = 512  #@param {type:'slider', min:256, max:512, step:128}  # info='MASt3R resizes so the long side equals this.'
LR1            = 0.07  #@param {type:'number'}  # info='MASt3R coarse GA learning rate. 0.07 = upstream default.'
NITER1         = 300  #@param {type:'slider', min:0, max:1000, step:50}  # info='MASt3R coarse GA iterations. 300 = upstream default.'
LR2            = 0.01  #@param {type:'number'}  # info='MASt3R refinement learning rate. 0.01 = upstream default.'
NITER2         = 300  #@param {type:'slider', min:0, max:1000, step:50}  # info='MASt3R refinement iterations. 0 = skip refinement. 300 = upstream default.'
MATCHING_CONF_THR = 5.0  #@param {type:'number'}  # info='Pairs below this confidence fall back to DUSt3R loss.'
ITERATIONS     = 7000  #@param {type:'slider', min:3000, max:30000, step:1000}  # info='3DGS training iterations. 7000 is a good balance.'
SH_DEGREE      = 3  #@param {type:'slider', min:0, max:3, step:1}  # info='SH degree for 3DGS. 3 = full view-dependent.'
ANTIALIASING   = False  #@param {type:'boolean'}  # info='Enable the Mip-NeRF 360 antialiasing filter.'
WHITE_BACKGROUND = False  #@param {type:'boolean'}  # info='Treat COLMAP background as white instead of black.'
VIDEO_FPS      = 2  #@param {type:'slider', min:1, max:5, step:1}  # info='Frames per second to extract from the video.'
DO_DRIVE_SAVE  = True  #@param {type:'boolean'}  # info='Copy outputs to /content/drive/MyDrive/AEI_3D_Out/WildGS/.'

if not INPUT_PATH.strip():
    raise SystemExit('Set INPUT_PATH to a video or a folder of images.')
src = pathlib.Path(INPUT_PATH).resolve()
if not src.exists():
    raise SystemExit(f'Input not found: {src}')
out = pathlib.Path(OUTPUT_DIR).resolve()
out.mkdir(parents=True, exist_ok=True)
ckpt = CKPT_PATH.strip() or None
print(f'  input    : {src}')
print(f'  output   : {out}')
print(f'  size     : {IMAGE_SIZE}')
print(f'  iters    : {ITERATIONS}')
print()
t0 = time.time()
ply = run_single_scene(
    src, out,
    image_size=IMAGE_SIZE, iterations=ITERATIONS,
    video_fps=VIDEO_FPS, do_drive_save=DO_DRIVE_SAVE,
    lr1=LR1, niter1=NITER1, lr2=LR2, niter2=NITER2,
    matching_conf_thr=MATCHING_CONF_THR,
    sh_degree=SH_DEGREE, antialiasing=ANTIALIASING,
    white_background=WHITE_BACKGROUND,
)
elapsed = time.time() - t0
print(f'\n  done in {elapsed:.0f}s')
print(f'  output: {ply}  ({ply.stat().st_size//(1024*1024)} MB)')


In [ ]:
#@title STEP 7 — Batch process multiple scenes
"""
Process every video or subdirectory in `INPUT_DIR` as a separate scene.
Each output scene has its own .ply + .mp4. Already-done scenes are skipped.
"""
import time, pathlib

INPUT_DIR      = ''  #@param {type:'string', placeholder: '/content/my_scenes/'}  # info='Folder containing videos or scene subdirs.'
OUTPUT_DIR     = '/content/WildGS_Out'  #@param {type:'string'}  # info='Where each scene's .ply and .mp4 will be written.'
CKPT_PATH      = ''  #@param {type:'string', placeholder: 'leave empty for default'}  # info='Override the MASt3R checkpoint path.'

IMAGE_SIZE     = 512  #@param {type:'slider', min:256, max:512, step:128}  # info='MASt3R resizes so the long side equals this.'
LR1            = 0.07  #@param {type:'number'}  # info='MASt3R coarse GA learning rate.'
NITER1         = 300  #@param {type:'slider', min:0, max:1000, step:50}  # info='MASt3R coarse GA iterations.'
LR2            = 0.01  #@param {type:'number'}  # info='MASt3R refinement learning rate.'
NITER2         = 300  #@param {type:'slider', min:0, max:1000, step:50}  # info='MASt3R refinement iterations.'
MATCHING_CONF_THR = 5.0  #@param {type:'number'}  # info='Pairs below this confidence fall back to DUSt3R loss.'
ITERATIONS     = 7000  #@param {type:'slider', min:3000, max:30000, step:1000}  # info='3DGS training iterations per scene.'
SH_DEGREE      = 3  #@param {type:'slider', min:0, max:3, step:1}  # info='SH degree for 3DGS.'
ANTIALIASING   = False  #@param {type:'boolean'}  # info='Enable the Mip-NeRF 360 antialiasing filter.'
WHITE_BACKGROUND = False  #@param {type:'boolean'}  # info='Treat COLMAP background as white instead of black.'
VIDEO_FPS      = 2  #@param {type:'slider', min:1, max:5, step:1}  # info='Frames per second to extract from videos.'
DO_DRIVE_SAVE  = True  #@param {type:'boolean'}  # info='Copy outputs to /content/drive/MyDrive/AEI_3D_Out/WildGS/.'

if not INPUT_DIR.strip():
    raise SystemExit('Set INPUT_DIR to a folder containing videos or scene subdirs.')
in_dir = pathlib.Path(INPUT_DIR).resolve()
if not in_dir.exists():
    raise SystemExit(f'Input not found: {in_dir}')
out_dir = pathlib.Path(OUTPUT_DIR).resolve()
out_dir.mkdir(parents=True, exist_ok=True)
ckpt = CKPT_PATH.strip() or None
print(f'  input    : {in_dir}')
print(f'  output   : {out_dir}')
print(f'  iters    : {ITERATIONS}')
print()
t0 = time.time()
n_ok = run_batch(
    str(in_dir), str(out_dir),
    image_size=IMAGE_SIZE, iterations=ITERATIONS,
    video_fps=VIDEO_FPS, do_drive_save=DO_DRIVE_SAVE,
    lr1=LR1, niter1=NITER1, lr2=LR2, niter2=NITER2,
    matching_conf_thr=MATCHING_CONF_THR,
    sh_degree=SH_DEGREE, antialiasing=ANTIALIASING,
    white_background=WHITE_BACKGROUND,
)
elapsed = time.time() - t0
print(f'\n  done in {elapsed:.0f}s  ({elapsed/max(1,n_ok):.0f}s per scene)')
